In [7]:
import os
import time
import urllib.request

import cv2
import mediapipe as mp

print("MediaPipe version:", getattr(mp, "__version__", "unknown"))

MediaPipe version: 1.2.0


In [8]:
os.makedirs("models", exist_ok=True)

MODEL_URLS = {
    "face": "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task",
    "pose": "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task",
    "hand": "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task",
}

def get_model(name, url):
    path = os.path.join("models", name + "_landmarker.task")

    if not os.path.isfile(path) or os.path.getsize(path) == 0:
        print(f"Downloading {name} model...")
        temp_path = path + ".download"
        try:
            urllib.request.urlretrieve(url, temp_path)
            os.replace(temp_path, path)
        finally:
            if os.path.exists(temp_path):
                os.remove(temp_path)

    return os.path.abspath(path)

face_model = get_model("face", MODEL_URLS["face"])
pose_model = get_model("pose", MODEL_URLS["pose"])
hand_model = get_model("hand", MODEL_URLS["hand"])

print("All model files are ready.")

All model files are ready.


In [ ]:

BaseOptions = mp.tasks.BaseOptions
Vision = mp.tasks.vision
RunningMode = Vision.RunningMode

# Face Detection - up to 10 faces
face_options = Vision.FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=face_model),
    running_mode=RunningMode.VIDEO,
    num_faces=10,
)

# Body/Pose Detection - up to 5 people
pose_options = Vision.PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=pose_model),
    running_mode=RunningMode.VIDEO,
    num_poses=5,
)

# Hand Detection - up to 10 hands
hand_options = Vision.HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=hand_model),
    running_mode=RunningMode.VIDEO,
    num_hands=10,
)

print("Multiple-person detection configured!")


Detector options configured.


In [10]:
POSE_CONNECTIONS = [
    (11, 12), (11, 13), (13, 15), (12, 14), (14, 16),
    (11, 23), (12, 24), (23, 24),
    (23, 25), (25, 27), (27, 29), (29, 31), (27, 31),
    (24, 26), (26, 28), (28, 30), (30, 32), (28, 32),
    (15, 17), (15, 19), (15, 21),
    (16, 18), (16, 20), (16, 22),
]

HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (5, 9), (9, 10), (10, 11), (11, 12),
    (9, 13), (13, 14), (14, 15), (15, 16),
    (13, 17), (0, 17), (17, 18), (18, 19), (19, 20),
]

# MediaPipe provides the face mesh connections in many Tasks API versions.
face_connection_class = getattr(Vision, 'FaceLandmarksConnections', None)
FACE_CONNECTIONS = (
    getattr(face_connection_class, 'FACE_LANDMARKS_TESSELATION', None)
    if face_connection_class is not None else None
)

print("Landmark connections configured.")

Landmark connections configured.


In [11]:
def draw_landmarks(
    image,
    landmarks,
    connections=None,
    color=(0, 255, 0),
    radius=2,
    point_step=1,
    line_thickness=1,
):
    if not landmarks:
        return

    height, width = image.shape[:2]
    points = []

    for landmark in landmarks:
        x = int(landmark.x * width)
        y = int(landmark.y * height)
        if 0 <= x < width and 0 <= y < height:
            points.append((x, y))
        else:
            points.append(None)

    if connections:
        for connection in connections:
            if hasattr(connection, 'start'):
                start, end = connection.start, connection.end
            else:
                start, end = connection

            if (
                0 <= start < len(points)
                and 0 <= end < len(points)
                and points[start] is not None
                and points[end] is not None
            ):
                cv2.line(
                    image, points[start], points[end], color,
                    line_thickness, cv2.LINE_AA
                )

    for index, point in enumerate(points):
        if point is not None and index % point_step == 0:
            cv2.circle(image, point, radius, color, -1)

print("Drawing function is ready.")

Drawing function is ready.


In [12]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    cap.release()
    raise RuntimeError(
        "Cannot open webcam. Check camera permissions, close other camera apps, "
        "or try changing VideoCapture(0) to VideoCapture(1)."
    )

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

start_time = time.monotonic()
frame_number = 0

try:
    with (
        Vision.FaceLandmarker.create_from_options(face_options) as face_detector,
        Vision.PoseLandmarker.create_from_options(pose_options) as pose_detector,
        Vision.HandLandmarker.create_from_options(hand_options) as hand_detector,
    ):
        print('Detection started. Press Q in the webcam window to stop.')

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret or frame is None:
                print('Could not read a webcam frame.')
                break

            frame = cv2.flip(frame, 1)
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=rgb_frame,
            )

            frame_number += 1
            timestamp_ms = max(
                frame_number,
                int((time.monotonic() - start_time) * 1000),
            )

            face_result = face_detector.detect_for_video(mp_image, timestamp_ms)
            pose_result = pose_detector.detect_for_video(mp_image, timestamp_ms)
            hand_result = hand_detector.detect_for_video(mp_image, timestamp_ms)

            # Face landmarks / mesh
            for face in face_result.face_landmarks:
                draw_landmarks(
                    frame, face, FACE_CONNECTIONS,
                    color=(80, 220, 80), radius=1,
                    point_step=2, line_thickness=1,
                )

            # Body pose
            for pose in pose_result.pose_landmarks:
                draw_landmarks(
                    frame, pose, POSE_CONNECTIONS,
                    color=(0, 165, 255), radius=3,
                    line_thickness=2,
                )

            # Both hands
            for hand in hand_result.hand_landmarks:
                draw_landmarks(
                    frame, hand, HAND_CONNECTIONS,
                    color=(255, 0, 255), radius=3,
                    line_thickness=2,
                )

            cv2.putText(
                frame, f'Faces: {len(face_result.face_landmarks)}',
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (80, 220, 80), 2
            )
            cv2.putText(
                frame, f'Poses: {len(pose_result.pose_landmarks)}',
                (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 165, 255), 2
            )
            cv2.putText(
                frame, f'Hands: {len(hand_result.hand_landmarks)}',
                (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 255), 2
            )

            cv2.imshow('Face, Pose and Hand Detection', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

finally:
    cap.release()
    cv2.destroyAllWindows()
    print('Detection stopped. Webcam released.')

Detection started. Press Q in the webcam window to stop.
Detection stopped. Webcam released.
